In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 14:12:52 WARN Utils: Your hostname, codespaces-9ec455, resolves to a loopback address: 127.0.0.1; using 10.0.1.161 instead (on interface eth0)
26/06/08 14:12:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 14:12:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 14:12:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


In [3]:
"""
Read data from students_offline.csv file and load into offline_students_raw table.
"""
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, ArrayType, MapType # type: ignore

offline_students_schema = StructType([
    StructField("ID", StringType()),
    StructField("FirstName", StringType()),
    StructField("LastName", StringType()),
    StructField("Address", StringType()),
    StructField("Skills", StringType()),
    StructField("Contacts", StringType())
])
# we load even complex data as string in the raw table

offline_students_df = spark.read.format("csv")\
                                .option("header", True)\
                                .option("quote", "\"")\
                                .option("escape", "\"")\
                                .schema(offline_students_schema)\
                                .load(path = "/workspaces/pyspark_udemy_codespace/data/students_offline.csv")

offline_students_df.show()
offline_students_df.printSchema()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{"AddressLine1":"...|[{"Skill":"Apache...|{"email":"xyz@abc...|
|102|    David|  Turner|{"AddressLine1":"...|[{"Skill":"Java",...|{"phone":"9873145...|
|103|    Katie|Mcloskey|{"AddressLine1":"...|[{"Skill":"SQL","...|{"email":"ert89@a...|
|104|   Nasima|  Khatun|{"AddressLine1":"...|[{"Skill":"Hadoop...|{"email":"magt23@...|
|105|   Pritam|    Jain|{"AddressLine1":"...|[{"Skill":"Python...|{"phone":"6984753...|
+---+---------+--------+--------------------+--------------------+--------------------+

root
 |-- ID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Skills: string (nullable = true)
 |-- Conta

In [4]:
offline_students_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("spark_db.offline_students_raw")
# overwriteSchema is used when we change the name of the columns or schema so we dont face error

26/06/08 14:13:43 ERROR Schema: Failed initialising database.
Database '/home/jovyan/work/metastore_db' not found.
org.datanucleus.exceptions.NucleusDataStoreException: Database '/home/jovyan/work/metastore_db' not found.
	at org.datanucleus.store.rdbms.ConnectionFactoryImpl$ManagedConnectionImpl.getConnection(ConnectionFactoryImpl.java:498)
	at org.datanucleus.store.rdbms.RDBMSStoreManager.<init>(RDBMSStoreManager.java:297)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:480)
	at org.datanucleus.plugin.NonManagedPluginRegistry.createExecutableExte

AnalysisException: org.apache.hadoop.hive.ql.metadata.HiveException: java.lang.RuntimeException: Unable to instantiate org.apache.hadoop.hive.ql.metadata.SessionHiveMetaStoreClient

In [ ]:
spark.sql("select * from spark_db.offline_students_raw").show()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{"AddressLine1":"...|[{"Skill":"Apache...|{"email":"xyz@abc...|
|102|    David|  Turner|{"AddressLine1":"...|[{"Skill":"Java",...|{"phone":"9873145...|
|103|    Katie|Mcloskey|{"AddressLine1":"...|[{"Skill":"SQL","...|{"email":"ert89@a...|
|104|   Nasima|  Khatun|{"AddressLine1":"...|[{"Skill":"Hadoop...|{"email":"magt23@...|
|105|   Pritam|    Jain|{"AddressLine1":"...|[{"Skill":"Python...|{"phone":"6984753...|
+---+---------+--------+--------------------+--------------------+--------------------+



In [ ]:
from pyspark.sql.functions import parse_json, col # type: ignore

offline_students_df = offline_students_df.withColumns({
    "Address": parse_json(col("Address")),
    "Skills": parse_json(col("Skills")),
    "Contacts": parse_json(col("Contacts"))
})
# parse_json is not there in my version of spark so it does not work here

offline_students_df.show()
offline_students_df.printSchema()
# the 3 columns should be of variant type
# varirant type means it could me struct, array or map but it doesnt matter

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_168/964058510.py", line 8, in <lambda>
  File "/opt/conda/lib/python3.11/json/__init__.py", line 339, in loads
    raise TypeError(f'the JSON object must be str, bytes or bytearray, '
TypeError: the JSON object must be str, bytes or bytearray, not dict


In [ ]:
# Variant object element names are case sensitive
# so to access an element in SQL we would use address:Country(this Country is CASE SENSITIVE)

"""
SELECT CAST(address:Country AS STRING), COUNT(*) AS total_count
FROM spark_db.offline_students
GROUP BY CAST(address:Country AS STRING)
"""

# We need to CAST variant type, it cannot be used in ORDER BY, GROUP BY and so on

"""
SELECT id, firstname, lastname, CAST(value:Skill as STRING), CAST(value:YearsOfExperience AS INTEGER)
FROM spark_db.offline_students, lateral VARIANT_EXPLODE(skills)
"""
# We use variant_explode() to explode variant data type but that cannot be done in SELECT clause but done in the FROM clause
# we CAST in the SELECT clause to not keep the data type as variant
# if for some student the skills is NULL, then it will not show in the result set. This is like an inner join
# to include such cases where we have null in the exploded column, we can use variant_explode_outer()

"""
SELECT id, firstname, lastname, contacts:email
FROM spark_db.offline_students
WHERE contacts:phone IS null AND contacts:whatsapp IS null
"""

'\nSELECT id, firstname, lastname, contacts:email\nFROM spark_db.offline_students\nWHERE contacts:phone IS null AND contacts:whatsapp IS null\n'